# Lesson 1.1 · Linear regression: the idea and least squares

This lesson turns lesson 4.1 of the theory course into code. We don't rederive the theory: we look at what the model
really does when you run it, and we build the tools you will use in the exercises.

**What you will learn**

- To read the prediction as a matrix product, ŷ = A·β, and to compute the coefficients by hand.
- To use `LinearRegression` and to read `coef_` and `intercept_` without getting the dimensions wrong.
- To understand why scikit-learn does not solve the normal equation, even though it is the textbook formula.
- To recognize when a straight line is the wrong choice.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression

## Definition

Linear regression models the target as a weighted sum of the features, plus a constant term called the
**intercept**. The weights, the **coefficients**, are chosen by minimizing the sum of squares of the **residuals**, that is
the differences between true value and predicted value (OLS criterion, Ordinary Least Squares).

```
ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₚxₚ          minimizes      Σᵢ (yᵢ − ŷᵢ)²
```

## Intuition: the line that minimizes the squares

Every vertical segment in the plot is a residual. Squaring them means that one error of 10 costs a hundred,
while ten errors of 1 cost ten in total: the line gets "pulled" toward the far points much more than
toward the near ones. That is the reason why outliers weigh so much (you will measure it in exercise C2).

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 25)
y = 2 + 1.5 * x + rng.normal(0, 2, 25)

model = LinearRegression().fit(x.reshape(-1, 1), y)
y_hat = model.predict(x.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(x, y, s=25)
grid = np.linspace(0, 10, 100)
ax.plot(grid, model.predict(grid.reshape(-1, 1)), color="#1F6E5C")
ax.vlines(x, y, y_hat, color="#9C4530", alpha=0.6, lw=1)        # the residuals
ax.set(xlabel="x", ylabel="y", title="The OLS line and its residuals (red segments)")

def ssr(b0, b1):
    return ((y - (b0 + b1 * x)) ** 2).sum()

print("SSR of the OLS line      :", round(ssr(model.intercept_, model.coef_[0]), 2))
print("SSR of an eyeballed line :", round(ssr(2, 1.3), 2))
print("SSR of another line      :", round(ssr(0, 1.8), 2))

## From the line to matrices

With p features it is convenient to write everything in matrix form. You build the matrix **A** by adding to X a
column of ones, which represents the intercept: so ŷ = A·β and the problem becomes
minimizing ‖y − A·β‖². Setting the gradient to zero gives the **normal equation**:

```
(AᵀA) β = Aᵀy        ⇒        β = (AᵀA)⁻¹ Aᵀy
```

In [ ]:
X, y = ds.diabetes()                                       # 442 patients, 10 features
A = np.column_stack([np.ones(len(X)), X.to_numpy()])      # column of ones + features
beta = np.linalg.solve(A.T @ A, A.T @ y.to_numpy())       # normal equation

sk = LinearRegression().fit(X, y)
pd.DataFrame({"by hand": beta, "sklearn": np.r_[sk.intercept_, sk.coef_]},
             index=["intercept", *X.columns]).round(3)

The two columns coincide. With a difference worth knowing: scikit-learn does **not** solve the normal
equation. It uses `scipy.linalg.lstsq`, which goes through the SVD decomposition, because computing AᵀA squares the
numerical conditioning of the matrix and, with nearly collinear features, loses almost all the
significant digits. You will see it with your own eyes in exercise A1.

## The scikit-learn API

The cycle is always the same: you create the object, call `fit(X, y)`, read the learned attributes (the ones
with a trailing underscore, by convention) and call `predict`.

In [ ]:
model = LinearRegression()
model.fit(X, y)

print("intercept:", round(model.intercept_, 2))
print("R2 on the training set:", round(model.score(X, y), 3))
pd.Series(model.coef_, index=X.columns).round(1)

In [ ]:
model.predict(X.iloc[:3])          # three patients, three predictions

> **Common mistake.** `fit` wants `X` to be **two-dimensional** (n_observations, n_features) and `y` one-dimensional. With a
> single feature a 1D array raises an error: you use `X[["column"]]` (double brackets) or
> `x.reshape(-1, 1)`.

In [ ]:
try:
    LinearRegression().fit(X["bmi"].to_numpy(), y)         # 1D array: wrong
except ValueError as error:
    print("ValueError:", str(error).splitlines()[0])

LinearRegression().fit(X[["bmi"]], y).coef_                # one-column DataFrame: correct

## When to use it and when to avoid it

- **Use it** as the baseline of every regression problem, when you need to interpret the effect of each variable
  and when the data are few compared to the number of features.
- **Avoid it** (or transform the features) when the relationship is clearly curved, when there are outliers you cannot
  remove, or when you must predict outside the range of the data seen: a line goes on forever.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why is there a column of ones in the matrix A?</summary>

Because the intercept is a coefficient like the others, multiplied by a feature that is always 1. By adding that column, the line is no longer forced to pass through the origin and the whole problem stays a single matrix product.

</details>

<details><summary>What changes with `fit_intercept=False` if the data are not centered?</summary>

The line is forced to pass through the origin. If the mean value of the target is far from zero, the model compensates by distorting the coefficients and the residuals no longer have mean zero: it is almost always a mistake.

</details>